# Práctica Apache Spark Standalone: Análisis de Acciones Bursátiles (`stocks.txt`)

- **Alumno:** René Nicolas Acosta López
- **Matrícula:** 125052937
- **Entorno:** BDA Lab (Docker: Hadoop + HDFS + Spark + Jupyter Notebook)

In [1]:
!mkdir -p DATOS
!rm -f ./DATOS/stocks.txt
!wget "https://github.com/josepgarcia/datos/raw/refs/heads/main/stocks.txt" -P ./DATOS
!head ./DATOS/stocks.txt

--2026-11-19 02:30:24--  https://github.com/josepgarcia/datos/raw/refs/heads/main/stocks.txt
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/stocks.txt [following]
--2026-11-19 02:30:25--  https://raw.githubusercontent.com/josepgarcia/datos/refs/heads/main/stocks.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 986 [text/plain]
Saving to: ‘./DATOS/stocks.txt’

stocks.txt          100%[===================>]     986  --.-KB/s    in 0s      

2026-11-19 02:30:25 (30,8 MB/s) - ‘./DATOS/stocks.txt’ saved [986/986]

id,name,category,quantity,price
1,iPhone,Electron

In [1]:
import os
import cloudpickle
import pyspark.serializers
pyspark.serializers.cloudpickle = cloudpickle

from pyspark.sql import SparkSession

# 1 y 2. Crear SparkSession con configuración
spark = SparkSession.builder \
    .appName("AnalisisStocks") \
    .config("spark.executor.memory", "512m") \
    .getOrCreate()

# 3. Obtener SparkContext
sc = spark.sparkContext

# 4. Construir ruta absoluta con prefijo file://
path_relativo = "./DATOS/stocks.txt"
path_absoluto = "file://" + os.path.abspath(path_relativo)

# 5. Cargar el archivo como RDD
rdd_raw = sc.textFile(path_absoluto)

# 6. Verificar la carga mostrando las primeras 5 líneas y particiones
print("Ruta cargada:", path_absoluto)
print("Número de particiones del RDD:", rdd_raw.getNumPartitions())
print("\nPrimeras 5 líneas:")
for linea in rdd_raw.take(5):
    print(linea)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/19 02:37:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Ruta cargada: file:///home/hadoop/notebooks/DATOS/stocks.txt
Número de particiones del RDD: 2

Primeras 5 líneas:


[Stage 0:>                                                          (0 + 1) / 1]

id,name,category,quantity,price
1,iPhone,Electronics,10,899.99
2,Macbook,Electronics,5,1299.99
3,iPad,Electronics,15,499.99
4,Samsung TV,Electronics,8,799.99


In [3]:
def parsear_linea(linea):
    try:
        partes = linea.strip().split(",")
        if len(partes) != 5:
            return None
        id_prod = int(partes[0])
        producto = partes[1].strip()
        categoria = partes[2].strip()
        cantidad = int(partes[3])
        precio = float(partes[4])
        return (id_prod, producto, categoria, cantidad, precio)
    except (ValueError, IndexError):
        return None

# Aplicar función a todo el RDD y filtrar líneas inválidas (como el encabezado)
rdd_parseado = rdd_raw.map(parsear_linea).filter(lambda x: x is not None)
rdd_parseado.cache()

# Contar líneas válidas
total_validas = rdd_parseado.count()
print(f"Total de líneas válidas: {total_validas:,}")
print("Ejemplo de tupla parseada:", rdd_parseado.first())

[Stage 1:=============================>                             (1 + 1) / 2]

Total de líneas válidas: 30
Ejemplo de tupla parseada: (1, 'iPhone', 'Electronics', 10, 899.99)


In [4]:
# 1. Crear pares (categoría, precio) -> índices 2 y 4
categoria_precio = rdd_parseado.map(lambda x: (x[2], x[4]))

# 2a. Precio máximo por categoría
precio_max = categoria_precio.reduceByKey(lambda a, b: max(a, b))

# 2b. Precio mínimo por categoría
precio_min = categoria_precio.reduceByKey(lambda a, b: min(a, b))

# 2c. Precio promedio por categoría usando aggregateByKey -> (suma, contador)
precio_promedio = categoria_precio.aggregateByKey(
    (0.0, 0),
    lambda acc, valor: (acc[0] + valor, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
).mapValues(lambda x: x[0] / x[1])

# 3. Unir resultados y ordenar por categoría
estadisticas_categoria = precio_max.join(precio_min).join(precio_promedio) \
    .mapValues(lambda x: (x[0][0], x[0][1], x[1])) \
    .sortByKey()

print("Estadísticas de precio por categoría (ordenado):")
print(f"{'Categoría':<18} | {'Máximo':>10} | {'Mínimo':>10} | {'Promedio':>10}")
print("-" * 57)
for cat, (pmax, pmin, pprom) in estadisticas_categoria.collect():
    print(f"{cat:<18} | ${pmax:9.2f} | ${pmin:9.2f} | ${pprom:9.2f}")

Estadísticas de precio por categoría (ordenado):
Categoría          |     Máximo |     Mínimo |   Promedio
---------------------------------------------------------
Accessories        | $    29.99 | $    24.99 | $    27.49
Clothing           | $   199.99 | $    14.99 | $    99.28
Electronics        | $  1299.99 | $   129.99 | $   586.66
Food               | $     5.99 | $     0.25 | $     2.30
Furniture          | $   249.99 | $    79.99 | $   141.99
Sports             | $    49.99 | $    19.99 | $    34.99


In [5]:
# 1. Crear pares (producto, cantidad) -> índices 1 y 3
producto_volumen = rdd_parseado.map(lambda x: (x[1], x[3]))

# 2. Sumar volumen (stock) por producto
volumen_por_producto = producto_volumen.reduceByKey(lambda a, b: a + b)

# 3. Ordenar por volumen descendente
top_productos = volumen_por_producto.sortBy(lambda x: x[1], ascending=False)

# 4. Mostrar top 10
print("Top 10 productos con mayor volumen en stock:")
for producto, cantidad in top_productos.take(10):
    print(f"{producto:<22}: {cantidad:,} unidades")

Top 10 productos con mayor volumen en stock:
Bananas               : 150 unidades
Oranges               : 120 unidades
Apples                : 100 unidades
T-shirt               : 50 unidades
Chicken Breast        : 50 unidades
Sneakers              : 40 unidades
Backpack              : 30 unidades
Nike Shoes            : 30 unidades
Salmon Fillet         : 30 unidades
Jeans                 : 30 unidades


In [6]:
# 1. Calcular (max, min) por categoría en una sola pasada
stats_por_categoria = categoria_precio.aggregateByKey(
    (float('-inf'), float('inf')),
    lambda acc, precio: (max(acc[0], precio), min(acc[1], precio)),
    lambda acc1, acc2: (max(acc1[0], acc2[0]), min(acc1[1], acc2[1]))
)

# 2. Calcular dispersión/volatilidad: ((max - min) / min) * 100
volatilidad = stats_por_categoria.mapValues(
    lambda x: ((x[0] - x[1]) / x[1]) * 100 if x[1] != 0 else 0.0
)

# 3 y 4. Identificar hasta 5 más volátiles y hasta 5 más estables
mas_volatiles = volatilidad.sortBy(lambda x: x[1], ascending=False).take(5)
mas_estables = volatilidad.sortBy(lambda x: x[1], ascending=True).take(5)

print("Categorías más volátiles (mayor dispersión de precio):")
for cat, vol in mas_volatiles:
    print(f"{cat:<18}: {vol:.2f}%")

print("\nCategorías más estables (menor dispersión de precio):")
for cat, vol in mas_estables:
    print(f"{cat:<18}: {vol:.2f}%")

Categorías más volátiles (mayor dispersión de precio):
Food              : 2296.00%
Clothing          : 1234.16%
Electronics       : 900.07%
Furniture         : 212.53%
Sports            : 150.08%

Categorías más estables (menor dispersión de precio):
Accessories       : 20.01%
Sports            : 150.08%
Furniture         : 212.53%
Electronics       : 900.07%
Clothing          : 1234.16%


In [7]:
# Crear carpeta RESULTADOS y limpiar corridas previas
!mkdir -p RESULTADOS
!rm -rf ./RESULTADOS/volumen_por_producto ./RESULTADOS/stats_por_categoria

# 1 y 2. Guardar resultados en archivos locales usando file://
output_path_volumen = "file://" + os.path.abspath("./RESULTADOS/volumen_por_producto")
volumen_por_producto.saveAsTextFile(output_path_volumen)

output_path_stats = "file://" + os.path.abspath("./RESULTADOS/stats_por_categoria")
stats_por_categoria.saveAsTextFile(output_path_stats)

# 3. Calcular métricas para el resumen ejecutivo
total_registros = rdd_parseado.count()
categorias_unicas = stats_por_categoria.count()
cat_mayor_precio = precio_promedio.sortBy(lambda x: x[1], ascending=False).first()
prod_mayor_volumen = top_productos.first()
cat_mas_volatil = mas_volatiles[0]

# Mostrar resumen ejecutivo
print("=" * 60)
print("RESUMEN EJECUTIVO")
print("=" * 60)
print(f"\nTotal de registros procesados     : {total_registros:,}")
print(f"Número de categorías únicas       : {categorias_unicas}")
print(f"\nCategoría con mayor precio prom.  : {cat_mayor_precio[0]} (${cat_mayor_precio[1]:.2f})")
print(f"Producto con mayor stock/volumen  : {prod_mayor_volumen[0]} ({prod_mayor_volumen[1]:,} unidades)")
print(f"Categoría más volátil en precios  : {cat_mas_volatil[0]} ({cat_mas_volatil[1]:.2f}%)")
print("=" * 60)

RESUMEN EJECUTIVO

Total de registros procesados     : 30
Número de categorías únicas       : 6

Categoría con mayor precio prom.  : Electronics ($586.66)
Producto con mayor stock/volumen  : Bananas (150 unidades)
Categoría más volátil en precios  : Food (2296.00%)


In [8]:
!ls -la ./RESULTADOS/volumen_por_producto ./RESULTADOS/stats_por_categoria
print("\n--- Muestra de volumen_por_producto ---")
!head -n 5 ./RESULTADOS/volumen_por_producto/part-00000
print("\n--- Muestra de stats_por_categoria ---")
!head -n 5 ./RESULTADOS/stats_por_categoria/part-00000

./RESULTADOS/stats_por_categoria:
total 28
drwxr-xr-x 2 hadoop hadoop 4096 Nov 19 02:49 .
drwxr-xr-x 4 hadoop hadoop 4096 Nov 19 02:49 ..
-rw-r--r-- 1 hadoop hadoop    8 Nov 19 02:49 ._SUCCESS.crc
-rw-r--r-- 1 hadoop hadoop   12 Nov 19 02:49 .part-00000.crc
-rw-r--r-- 1 hadoop hadoop   12 Nov 19 02:49 .part-00001.crc
-rw-r--r-- 1 hadoop hadoop    0 Nov 19 02:49 _SUCCESS
-rw-r--r-- 1 hadoop hadoop  120 Nov 19 02:49 part-00000
-rw-r--r-- 1 hadoop hadoop   58 Nov 19 02:49 part-00001

./RESULTADOS/volumen_por_producto:
total 28
drwxr-xr-x 2 hadoop hadoop 4096 Nov 19 02:49 .
drwxr-xr-x 4 hadoop hadoop 4096 Nov 19 02:49 ..
-rw-r--r-- 1 hadoop hadoop    8 Nov 19 02:49 ._SUCCESS.crc
-rw-r--r-- 1 hadoop hadoop   12 Nov 19 02:49 .part-00000.crc
-rw-r--r-- 1 hadoop hadoop   12 Nov 19 02:49 .part-00001.crc
-rw-r--r-- 1 hadoop hadoop    0 Nov 19 02:49 _SUCCESS
-rw-r--r-- 1 hadoop hadoop  344 Nov 19 02:49 part-00000
-rw-r--r-- 1 hadoop hadoop  206 Nov 19 02:49 part-00001

--- Muestra de volumen_por_

In [9]:
spark.stop()
print("\n✅ Análisis completado. Sesión cerrada.")


✅ Análisis completado. Sesión cerrada.


> **Nota metodológica:** El archivo oficial `stocks.txt` alojado en `josepgarcia/datos` contiene 30 registros de inventario de productos con el esquema `id,name,category,quantity,price` en lugar de cotizaciones bursátiles. El análisis se adaptó fielmente a este esquema real conservando todas las transformaciones, agregaciones y acciones solicitadas en los 6 pasos de la práctica.

## Preguntas de Reflexión

1. **Eficiencia (`split()` una sola vez por línea):**
   Ejecutar `split(",")` una sola vez guarda la lista de campos en memoria y permite indexar cada columna en tiempo constante $O(1)$. Hacer `split()` por separado para cada campo obligaría a escanear la cadena 5 veces y crear 5 listas temporales por línea; en 1 millón de líneas se pasa de 1 millón a 5 millones de operaciones de parseo y asignaciones de memoria, consumiendo mucha más CPU y ciclos de recolección de basura.

2. **Particionamiento:**
   El RDD se creó con **2 particiones** (verificado con `rdd_raw.getNumPartitions()`). Cada partición se ejecuta como una tarea independiente en paralelo. Con solo 1 partición se perdería todo el paralelismo: un único hilo haría todo el trabajo mientras los demás núcleos quedan ociosos, creando un cuello de botella y mayor riesgo de agotar la memoria en esa tarea.

3. **Lazy Evaluation:**
   Las transformaciones (`map`, `filter`, `reduceByKey`, `aggregateByKey`, `sortBy`, `mapValues`) no se ejecutaron al definirse, sino hasta invocar acciones (`take`, `count`, `collect`, `first`, `saveAsTextFile`). La ventaja es que Spark inspecciona el grafo completo de operaciones (DAG) antes de ejecutarlo, pudiendo encadenar transformaciones en una sola pasada (*pipelining*), optimizar lecturas y evitar trabajo innecesario.

4. **Memoria (`collect()` en un RDD de 10 GB):**
   `collect()` reúne todos los datos de los *executors* en la memoria RAM del nodo *Driver*. Con 10 GB de datos (más el peso de los objetos en Python), el *Driver* colapsaría con `OutOfMemoryError` (OOM). Se evita agregando o filtrando de forma distribuida antes de traer resultados, usando `take(n)` para muestras pequeñas o guardando directo en almacenamiento distribuido con `saveAsTextFile()`.

5. **Optimización para 100x más datos:**
   - Cachear RDDs reutilizados (`rdd_parseado.cache()` o `.persist(StorageLevel.MEMORY_AND_DISK)`) para evitar releer y reparsear desde disco en cada acción.
   - Ajustar el número de particiones con `repartition()` para mantener entre 2 y 4 particiones por núcleo disponible.
   - Calcular máximo, mínimo y promedio en una sola pasada de `aggregateByKey`, o migrar a **Spark DataFrames** para aprovechar el optimizador Catalyst y evitar la sobrecarga de serialización entre JVM y Python.

6. **HDFS vs. Local:**
   En un clúster multinodo, los archivos locales (`file://`) obligan a copiar manualmente el archivo en la misma ruta de cada nodo y están limitados a un solo disco. HDFS divide los archivos en bloques distribuidos y replicados entre nodos, ofreciendo tolerancia a fallos, escalabilidad horizontal para grandes volúmenes y localidad de datos (Spark corre el cómputo en los nodos donde residen los bloques, reduciendo el tráfico de red).